# Libraries

In [1]:
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors
!pip install --upgrade transformers #FIX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 86.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 73.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━

In [2]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [4]:
model_name = "EdgeCompress01/Qwen3-4B-SparseGPT"
subfolder = "Models/70"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [7]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [8]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    model_args = f"pretrained={model_name},subfolder={subfolder},max_length=2048,dtype=float16,parallelize=True"
    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", cfg["task_name"],
        "--batch_size", str(3),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]

    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-27:13:43:29 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:13:43:35 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 314441.68 examples/s]
2026-03-27:13:44:26 WARNING  [evaluator:333] Overwriting default num_fewshot of gsm8k_cot from 8 to 2
Running generate_until requests: 100%|██████████| 100/100 [10:37<00:00,  6.38s/it]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Qwen3-4B-SparseGPT', 'subfolder': 'Models/70', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|---------|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     2|exact_match|↑  | 0.01|±  |  0.01|
|         |       |strict-match    |     2|exact_match|↑  | 0.00|±  |  0.00|

Finished gsm8k

Starting evaluation for: mmlu


2026-03-27:13:55:11 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:13:55:16 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 2916.36 examples/s]
2026-03-27:13:56:52 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 2
2026-03-27:13:56:52 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 2
2026-03-27:13:56:52 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 2
2026-03-27:13:56:52 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 2
2026-03-27:13:56:52 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 2
2026-03-27:13:56:52 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_computer_science from None to 2
2026-03-27:1

hf ({'pretrained': 'EdgeCompress01/Qwen3-4B-SparseGPT', 'subfolder': 'Models/70', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.2407|±  |0.0057|
| - humanities                          |      2|none  |      |acc   |↑  |0.2508|±  |0.0120|
|  - formal_logic                       |      1|none  |     2|acc   |↑  |0.1500|±  |0.0359|
|  - high_school_european_history       |      1|none  |     2|acc   |↑  |0.3000|±  |0.0461|
|  - high_school_us_history             |      1|none  |     2|acc   |↑  |0.2600|±  |0.0441|
|  - high_school_world_history          |      1|none  |     2|acc   |↑  |0.2200|±  |0.0416|
|  - international_law         Finished mmlu

Sta

2026-03-27:14:16:23 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:14:16:28 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
Generating validation split: 100%|██████████| 299/299 [00:00<00:00, 103893.37 examples/s]
2026-03-27:14:16:42 WARNING  [evaluator:333] Overwriting default num_fewshot of arc_challenge from None to 0
Running loglikelihood requests: 100%|██████████| 400/400 [00:11<00:00, 33.54it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


Finished arc_challenge

Starting evaluation for: wikitext


2026-03-27:14:16:59 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:14:17:04 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
Loading weights: 100%|██████████| 398/398 [00:02<00:00, 190.32it/s]
2026-03-27:14:17:16 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-03-27:14:17:16 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-03-27:14:17:16 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-03-27:14:17:16 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-03-27:14:17:16 WARNING  [api.task:729] [Task: wiki

Finished wikitext



# Save reports

In [9]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
FileLink(zip_path)

/kaggle/working/evaluation_results.zip